# LIBERO eval — **최종 표** (상태판단 + SR + 떨림[aloha 방식])

실행하면 **① 학습/eval 됐는지 먼저 판단** → ② 500ep 완료만으로 SR·떨림 표 → ③ zip.

- **떨림은 팀원(은지) aloha 스크립트와 동일 계산식**(`smooth_metrics_paper`): 경계/내부 jerk RMS,
  B/I ratio, SPARC(speed-profile, fs=30), ldj_cost, sign-flip rate. → aloha 와 바로 비교 가능.
- 표기: `bimamba_s7` → **ours**, 이름의 `s7` → **mosaic**.
- **MOSAIC = carry + crossfade(한 묶음)** → ablation 은 `mosaic only`(bimamba 없음)·`bimamba only`(crossfade 없음) 2개.
- 읽기 전용. ①의 결과를 나(클로드)한테 주면 **학습할 것 / (재)eval 할 것** 을 정리해줌.


In [ ]:
import sys, json, csv, time
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
import numpy as np
import smooth_metrics_paper as smp        # 팀원(aloha) 과 동일한 스무스니스 계산식
importlib.reload(smp)

TASK   = 'libero_10'
SEEDS  = [0, 1, 2, 3]
TARGET_EP = cf.EVAL_N_EP                 # 500
STRICT_EP = True                         # True=정확히 500ep 인 eval 만(5000·50 등은 제외+재eval 표시)
FS     = cf.fps_of(TASK)                 # libero=30 (SPARC 주파수축에만 영향)
STRIDE = 100                             # 청크 경계(하드스위치 기준). aloha 와 동일

# ── 최종 표 모델 (folder_tag, 표기 라벨, 역할) ─  s7→mosaic, bimamba_s7→ours ─
#   ※ MOSAIC = carry + crossfade(한 묶음). 그래서 ablation 은 {mosaic only, bimamba only} 2개.
#     - mosaic only  = carry+crossfade, bimamba 없음   (미학습 예상 → '학습 필요' 로 뜸)
#     - bimamba only = carry+BiMamba,   crossfade 없음 (= 학습된 'bimamba' 폴더)
#     - ours         = carry+crossfade+BiMamba          (= 'bimamba_s7')
DESIRED = [
    ('act',        'ACT',                     'baseline'),
    ('acm',        'ACM (Mamba-1, no carry)', 'baseline'),
    ('acm2',       'ACM2 (Mamba-2, no carry)','baseline'),
    ('mosaic',     'mosaic only',             'ablation'),   # carry+crossfade (폴더명 미확정, 미학습 예상)
    ('bimamba',    'BiMamba only',            'ablation'),   # carry+BiMamba (= 학습된 bimamba)
    ('bimamba_s7', 'ours',                    'full'),
]
def label(tag):
    for t, lb, _ in DESIRED:
        if t == tag:
            return lb
    return 'ours' if tag == 'bimamba_s7' else tag.replace('s7', 'mosaic')

for t, _, _ in DESIRED:
    cf.v23.MODEL_DIR_NAMES.setdefault(t, t)   # 비표준 태그도 폴더명=태그로 등록

TRAIN_ROOT = cf.OUTPUT_BASE / 'train' / TASK
EVAL_ROOT  = cf.OUTPUT_BASE / 'eval_clean' / TASK
OUT = cf.OUTPUT_BASE / 'share' / f'{TASK}_final'
OUT.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime('%Y%m%d_%H%M')

# 셀에 찍힌 표와 같은 모양을 이미지로도 저장하는 공용 헬퍼
def save_table_png(col_labels, cell_text, title, path, colw=None):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    nrow, ncol = len(cell_text), len(col_labels)
    fig, ax = plt.subplots(figsize=(1.4 + 1.25 * ncol, 0.6 + 0.42 * (nrow + 1)))
    ax.axis('off')
    tb = ax.table(cellText=cell_text, colLabels=col_labels, loc='center', cellLoc='center')
    tb.auto_set_font_size(False); tb.set_fontsize(10)
    tb.auto_set_column_width(col=list(range(ncol)))   # 열 너비를 내용에 맞춰(라벨 잘림 방지)
    tb.scale(1, 1.5)
    for (r, c), cell in tb.get_celld().items():
        if r == 0:
            cell.set_text_props(weight='bold'); cell.set_facecolor('#e8e8e8')
        if c == 0 and r > 0:
            cell.set_text_props(ha='left'); cell.PAD = 0.04
    ax.set_title(title, fontsize=12, pad=10)
    fig.savefig(path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print('  이미지 저장:', path)

print('train:', TRAIN_ROOT, '| eval:', EVAL_ROOT)
print(f'fs={FS} (SPARC) · 경계 stride={STRIDE} · 최종 채택 {TARGET_EP}ep · 스냅샷 {STAMP}')

## 1) 상태 판단 — 학습됐나 / 500ep eval 됐나 / action 있나
여기서 **학습 필요 / (재)eval 필요 / 준비됨** 으로 자동 분류. carry-only·overlap-only·acm2 는 대개 미학습으로 뜬다.


In [ ]:
# ── 상태 판단: 각 모델×seed 가 학습됐나 / 500ep eval 됐나 / action(.pt) 있나 ──
def eval_rec(tag, seed):
    d = EVAL_ROOT / tag / f'seed{seed}'          # rep0/ 등 하위 포함
    if not d.is_dir():
        return None
    recs = []
    for info in d.rglob('eval_info.json'):
        try:
            ov = json.loads(info.read_text()).get('overall', {})
        except Exception:
            continue
        n_ep = ov.get('n_ep', ov.get('n_episodes')) or 0
        recs.append({'sr': ov.get('pc_success'), 'n_ep': n_ep,
                     'has_actions': (info.parent / 'actions').is_dir(), 'path': info.parent})
    if not recs:
        return None
    exact = [r for r in recs if r['n_ep'] == TARGET_EP]     # 정확히 500ep 우선
    pool = exact if exact else recs
    best = sorted(pool, key=lambda r: (r['has_actions'], r['n_ep']))[-1]  # action 있는 것·큰 것
    best['exact'] = (best['n_ep'] == TARGET_EP)             # 500ep 정확히 맞나
    return best

def trained(tag, seed):
    return cf.v23.last_ckpt_step(TRAIN_ROOT / tag / f'seed{seed}')

print(f"{'model (라벨)':<26}{'seed':>5}{'학습':>10}{'eval':>10}{'action':>8}")
print('-' * 60)
todo_train, todo_eval, ready = [], [], []
status = {}
for tag, lb, role in DESIRED:
    n_tr = n_ev = n_act = 0
    for s in SEEDS:
        st = trained(tag, s); ev = eval_rec(tag, s)
        ok_tr = st is not None and st >= cf.CKPT_STEP
        ok_ev = ev is not None and (ev['n_ep'] or 0) >= TARGET_EP
        ok_act = bool(ev and ev['has_actions'])
        n_tr += ok_tr; n_ev += ok_ev; n_act += ok_act
        tr_s = f'{st:,}' if st else '-'
        ev_s = f"{ev['n_ep']}ep" if ev else '-'
        print(f"{tag+' ('+lb+')':<26}{s:>5}{tr_s:>10}{ev_s:>10}{'O' if ok_act else '-':>8}")
    status[tag] = {'tr': n_tr, 'ev': n_ev, 'act': n_act}
    if n_tr == 0:
        todo_train.append(tag)
    elif n_ev < len(SEEDS) or n_act < n_ev:
        todo_eval.append(tag)
    else:
        ready.append(tag)

print('\n' + '=' * 60)
print('■ 학습 필요 (체크포인트 없음):        ', [f'{t}({label(t)})' for t in todo_train] or '없음')
print('■ eval/재eval 필요 (학습됐으나 500ep·action 부족):', [f'{t}({label(t)})' for t in todo_eval] or '없음')
print('■ 준비됨 (500ep+action, 표에 반영):   ', [f'{t}({label(t)})' for t in ready] or '없음')

# 폴더에 있는데 DESIRED 에 없는 모델(누락 방지)
on_disk = {p.name for p in TRAIN_ROOT.glob('*') if p.is_dir()} | {p.name for p in EVAL_ROOT.glob('*') if p.is_dir()}
extra = sorted(on_disk - {t for t, _, _ in DESIRED})
if extra:
    print('\n(참고) DESIRED 밖 폴더도 있음:', extra)

## 2) SR 표 (500ep 완료만, mean±std, pooled 95% CI) — 셀 출력 + 이미지 저장


In [ ]:
# ── SR 표: 500ep 완료만 · model×seed · mean±std · pooled 95%CI (라벨 적용) ──
hdr = f"{'model':<26}" + ''.join(f'{("s"+str(s)):>7}' for s in SEEDS) + f"{'mean':>8}{'±std':>7}{'ep':>8}{'pooled95%CI':>16}"
print(hdr); print('-' * len(hdr))
sr_rows = []
sr_epinfo, reeval = {}, []
for tag, lb, role in DESIRED:
    per, eps, pk, pn = {}, {}, 0, 0
    for s in SEEDS:
        ev = eval_rec(tag, s)
        if not (ev and ev['sr'] is not None and (ev['n_ep'] or 0) >= TARGET_EP):
            continue
        if STRICT_EP and not ev['exact']:
            reeval.append((lb, s, ev['n_ep']))       # 500ep 아님 → 표에서 빼고 재eval 목록
            continue
        per[s] = ev['sr']; n = int(ev['n_ep']); eps[s] = n
        pk += int(round(ev['sr'] / 100 * n)); pn += n
    sr_epinfo[tag] = eps
    vals = list(per.values())
    mean = float(np.mean(vals)) if vals else None
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else (0.0 if vals else None)
    ci = ''
    if pn:
        lo, hi = cf.v23.wilson_ci(pk, pn); ci = f'[{lo*100:.1f},{hi*100:.1f}]'
    cells = ''.join((f'{per[s]:>7.1f}' if s in per else f'{chr(183):>7}') for s in SEEDS)
    mean_s = f'{mean:>8.1f}' if mean is not None else f'{chr(45):>8}'
    std_s = f'{std:>7.1f}' if std is not None else f'{chr(45):>7}'
    print(f'{lb:<26}{cells}{mean_s}{std_s}{pn:>8}{ci:>16}')
    sr_rows.append({'model': tag, 'label': lb, 'role': role,
                    **{f'seed{s}': (round(per[s], 1) if s in per else None) for s in SEEDS},
                    'mean': (round(mean, 2) if mean is not None else None),
                    'std': (round(std, 2) if std is not None else None),
                    'n_seed': len(vals), 'pooled_n': pn,
                    'pooled_sr': (round(pk / pn * 100, 2) if pn else None), 'pooled_ci': ci})
cols = ['model', 'label', 'role'] + [f'seed{s}' for s in SEEDS] + ['mean', 'std', 'n_seed', 'pooled_n', 'pooled_sr', 'pooled_ci']
with open(OUT / f'sr_table_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(sr_rows)
print('\nsaved:', OUT / f'sr_table_{STAMP}.csv')

# seed 별 ep(에피소드) 개수 — 500 인지 확인용
print('\nep 개수(seed별 eval n_episodes):')
for tag, lb, role in DESIRED:
    eps = sr_epinfo.get(tag) or {}
    if eps:
        print(f'   {lb:<26} ' + '  '.join(f'seed{s}:{eps[s]}' for s in SEEDS if s in eps)
              + f'   (합 {sum(eps.values())})')

if reeval:
    print(f'\n⚠️ 500ep 아님 → 표에서 제외, **재eval 필요** ({len(reeval)}개):')
    for lb, s, n in reeval:
        print(f'   {lb:<26} seed{s}  현재 {n}ep  → 옛 eval 지우고 500ep 재실행')
    print('   (STRICT_EP=False 로 두면 500 이상도 그대로 사용)')

# 위 표와 같은 모양 이미지로 저장 (ep 합계 열 포함)
img_col = ['model'] + [f'seed{s}' for s in SEEDS] + ['mean', '±std', 'n_seed', 'ep(합)', 'pooled 95%CI']
img_txt = []
for r in sr_rows:
    row = [r['label']]
    row += [(f"{r['seed'+str(s)]:.1f}" if r['seed' + str(s)] is not None else '·') for s in SEEDS]
    row += ['-' if r['mean'] is None else f"{r['mean']:.1f}",
            '-' if r['std'] is None else f"{r['std']:.1f}",
            f"{r['n_seed']}/{len(SEEDS)}", str(r['pooled_n']), r['pooled_ci'] or '-']
    img_txt.append(row)
save_table_png(img_col, img_txt, f'LIBERO-10 SR ({TARGET_EP}ep)   {STAMP}', OUT / f'sr_table_{STAMP}.png')

## 3) 떨림 표 — aloha 와 동일 (경계/내부 jerk · SPARC · ldj_cost · sign-flip) — 셀 출력 + 이미지 저장


In [ ]:
# ── 떨림 표 (팀원=aloha 방식): 경계/내부 jerk RMS · B/I ratio · SPARC · ldj_cost · signflip ──
#   500ep 완료 + action(.pt) 있는 seed 의 궤적을 pool. fs=30, 경계 stride=100.
#   ⚠️ 옛 50ep eval 의 action 이 섞이는 문제(예: seed1 = 500 + 옛 50): 궤적을 **오래된 순**으로 정렬해
#      500 의 배수가 되도록 **가장 오래된 나머지(count % 500)** 만 버린다. 500·5000 등은 그대로 유지.
import glob, os
def _load_aged(actions_dir):
    import numpy as np
    out, ad = [], Path(actions_dir)          # (mtime, name, idx, traj)
    if not ad.is_dir():
        return out
    def add(arr, mt, name):
        arr = np.asarray(arr)
        if arr.ndim == 3:
            for i in range(arr.shape[0]): out.append((mt, name, i, arr[i]))
        elif arr.ndim == 2:
            out.append((mt, name, 0, arr))
    for p in sorted(glob.glob(str(ad / '*.np[yz]'))):
        try: d = np.load(p, allow_pickle=True)
        except Exception: continue
        mt = os.path.getmtime(p)
        if isinstance(d, np.lib.npyio.NpzFile):
            arr = next((d[k] for k in ('actions', 'action', 'arr_0') if k in d), None)
            if arr is None and len(d.files): arr = d[d.files[0]]
            if arr is not None: add(arr, mt, os.path.basename(p))
        else:
            add(d, mt, os.path.basename(p))
    ptd = ad / 'action_logs'
    if ptd.is_dir():
        import torch
        for p in sorted(glob.glob(str(ptd / '*.pt'))):
            try: t = torch.load(p, map_location='cpu')
            except Exception: continue
            if hasattr(t, 'detach'): t = t.detach().cpu().numpy()
            add(t, os.path.getmtime(p), os.path.basename(p))
    return out

def _trim_old(aged, block):
    aged = sorted(aged, key=lambda x: (x[0], x[1], x[2]))   # 오래된 순
    drop = len(aged) % block                                 # 500 배수 안 되는 나머지 = 옛 데이터
    return [t for _, _, _, t in aged[drop:]], drop           # 가장 오래된 나머지 제거

smooth, smooth_seedinfo = {}, {}
for tag, lb, role in DESIRED:
    trajs, used = [], []
    for s in SEEDS:
        ev = eval_rec(tag, s)
        if not (ev and (ev['n_ep'] or 0) >= TARGET_EP and ev['has_actions']):
            continue
        if STRICT_EP and not ev['exact']:      # 500ep 아닌 seed 는 떨림도 제외(재eval 대상)
            continue
        kept, dropped = _trim_old(_load_aged(ev['path'] / 'actions'), TARGET_EP)
        used.append(f'seed{s}:{len(kept)}' + (f'(옛{dropped}제거)' if dropped else ''))
        trajs += kept
    smooth_seedinfo[tag] = used
    if trajs:
        smooth[tag] = smp.aggregate_paper(trajs, boundary_stride=STRIDE, fs=FS)

print('떨림용 궤적 수(seed별, 옛N제거=가장 오래된 것 버림):')
for tag, lb, role in DESIRED:
    if smooth_seedinfo.get(tag):
        print(f'   {lb:<24} {"  ".join(smooth_seedinfo[tag])}')
print()

if not smooth:
    print('action(.pt) 있는 완료 eval 이 없음 — SR 표만. (eval 이 RECORD_DIR 로 궤적 저장했는지 확인)')
else:
    print(f"{'model':<24}{'jerk_RMS':>9}{'bnd_jerk':>9}{'int_jerk':>9}{'B/I':>7}{'SPARC':>9}{'ldj_cost':>9}{'signflip':>9}{'n':>6}")
    print('  방향:            ↓        ↓        ↓     →1     →0        ↓        ↓')
    print('-' * 91)
    srows = []
    for tag, lb, role in DESIRED:
        a = smooth.get(tag)
        if not a:
            continue
        print(f"{lb:<24}{a['jerk_rms_mean']:>9.4f}{a['boundary_jerk_rms_mean']:>9.4f}"
              f"{a['interior_jerk_rms_mean']:>9.4f}{a['boundary_interior_ratio_mean']:>7.2f}"
              f"{a['sparc_mean']:>9.2f}{a['ldj_cost_mean']:>9.2f}{a['sign_flip_rate_mean']:>9.4f}{a['n_traj']:>6}")
        srows.append({'model': tag, 'label': lb,
                      'jerk_rms': round(a['jerk_rms_mean'], 5),
                      'boundary_jerk_rms': round(a['boundary_jerk_rms_mean'], 5),
                      'interior_jerk_rms': round(a['interior_jerk_rms_mean'], 5),
                      'boundary_interior_ratio': round(a['boundary_interior_ratio_mean'], 4),
                      'sparc': round(a['sparc_mean'], 3), 'ldj_cost': round(a['ldj_cost_mean'], 3),
                      'sign_flip_rate': round(a['sign_flip_rate_mean'], 5), 'n_traj': a['n_traj']})
    with open(OUT / f'smoothness_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'label', 'jerk_rms', 'boundary_jerk_rms',
            'interior_jerk_rms', 'boundary_interior_ratio', 'sparc', 'ldj_cost', 'sign_flip_rate', 'n_traj'])
        w.writeheader(); w.writerows(srows)
    print('\nsaved:', OUT / f'smoothness_{STAMP}.csv')

    # 위 표와 같은 모양 이미지로 저장 (방향 표기 헤더 포함)
    img_col = ['model', 'jerk_RMS↓', 'bnd_jerk↓', 'int_jerk↓', 'B/I→1', 'SPARC→0', 'ldj_cost↓', 'signflip↓', 'n']
    img_txt = [[r['label'], f"{r['jerk_rms']:.4f}", f"{r['boundary_jerk_rms']:.4f}",
               f"{r['interior_jerk_rms']:.4f}", f"{r['boundary_interior_ratio']:.2f}",
               f"{r['sparc']:.2f}", f"{r['ldj_cost']:.2f}", f"{r['sign_flip_rate']:.4f}", str(r['n_traj'])]
               for r in srows]
    save_table_png(img_col, img_txt, f'LIBERO-10 smoothness (aloha-matched, fs={FS})   {STAMP}',
                   OUT / f'smoothness_{STAMP}.png')

## 4) 요약 MD + zip (위 표 이미지·CSV 포함)


In [ ]:
# ── 요약 MD + zip (SR/떨림 표 이미지·CSV 는 위 셀에서 이미 저장됨) ──
lines = [f'# LIBERO-10 최종 ({TARGET_EP}ep, {STAMP})', '',
         f'- 떨림 = aloha 와 동일 스크립트(smooth_metrics_paper), fs={FS}, 경계 stride={STRIDE}',
         f'- 학습 필요: {todo_train}', f'- eval/재eval 필요: {todo_eval}', f'- 준비됨: {ready}']
(OUT / f'README_{STAMP}.md').write_text('\n'.join(lines), encoding='utf-8')
import shutil
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'{TASK}_final_{STAMP}'), 'zip', root_dir=OUT)
print('보낼 파일:', zip_path)
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')